In this cell I want to test an RPE-aware, reward-aware, PM-aware, and null DDM model of recognition reaction time starting with the Prolific data. 

In [ ]:
import pandas as pd
import numpy as np

# Get prolific data: 

base_dir = '/Volumes/T7 Shield' # this is the root directory for most un-archived data and results
load_dir = f'{base_dir}/work/qasims01/MemoryBanditData/Prolific'
save_dir_plots = '/Users/salmanqasim/Documents/Projects/Sinai/MemBanditsEMU/Plots'


In [ ]:
# load behavioral data from Prolific subjects
learn_df = pd.read_csv(f'{load_dir}/DelayedMemoryTest/learn_df_HierarchicalRW.csv')
combined_df = pd.read_csv(f'{load_dir}/DelayedMemoryTest/full_data_df_HierarchicalRW.csv')


In [ ]:
learn_df['gamble_rt_log'] = np.log(learn_df.gamble_rt)

combined_df['gamble_rt_log'] = np.log(combined_df.gamble_rt)
combined_df['recog_rt_log'] = np.log(combined_df.recog_rt)



In [ ]:
combined_df['choice'] = combined_df['recog_choice'].map({'old': 1, 'new': 0})
combined_df.rename(columns={'recog_rt':'rt'}, inplace=True)


In [ ]:
# learn_df = learn_df[['participant', 'trials', 'choice', 'reward', 'rt']]

# Drop bad trials
combined_df = combined_df.dropna(subset=["choice", "reward", "rt", "rpe", "image_dprime"])
combined_df = combined_df[combined_df.rt > 0.25]      # remove anticipatory RTs
combined_df = combined_df[combined_df.rt < 10.0]       # remove outliers

# Convert participants to indices
combined_df["subj_idx"] = pd.factorize(combined_df["participant"])[0] + 1

In [ ]:
combined_df = combined_df[combined_df.condition == 'day_1']

In [ ]:
combined_df

In [ ]:
# Per-subject minimum RT (same order as subject indices 1..S) for t0 constraint
min_rt_subj = combined_df.groupby("subj_idx")["rt"].min().sort_index().values.astype(float)

base_stan_data = {
    "N": len(combined_df),
    "S": int(combined_df.subj_idx.nunique()),
    "subj": combined_df.subj_idx.values.astype(int),
    "choice": combined_df.choice.values.astype(int),
    "rt": combined_df.rt.values.astype(float),
    "min_rt_subj": min_rt_subj,
    "max_a": 10.0,
}

covariate_data = {
    "reward": combined_df.reward.values.astype(float),
    "rpe": combined_df.rpe.values.astype(float),
    "pm": combined_df.image_dprime.values.astype(float),
}


def build_stan_data(extra_keys):
    """Subset dict per Stan model — CmdStanPy errors on unused keys."""
    d = dict(base_stan_data)
    for k in extra_keys:
        d[k] = covariate_data[k]
    return d


ddm_specs = [
    {"key": "vanilla", "stan_file": "recog_ddm.stan", "extra_keys": ()},
    {"key": "reward_drift", "stan_file": "recog_ddm_reward.stan", "extra_keys": ("reward",)},
    {"key": "rpe_drift", "stan_file": "recog_ddm_rpe.stan", "extra_keys": ("rpe",)},
    {"key": "pm_drift", "stan_file": "recog_ddm_pm.stan", "extra_keys": ("pm",)},
]

# Full merged dict for ad-hoc checks / legacy references
stan_data = build_stan_data(("reward", "rpe", "pm"))

In [ ]:
from pathlib import Path

from cmdstanpy import CmdStanModel
import arviz as az

# Run this notebook with cwd = Notebooks/ (same directory as the .stan files)
STAN_DIR = Path(".").resolve()
FITS_DIR = STAN_DIR / "fits"
FITS_DIR.mkdir(exist_ok=True)

SAMPLER_KWDS = dict(
    chains=4,
    parallel_chains=4,
    iter_warmup=1000,
    iter_sampling=1000,
    adapt_delta=0.95,
    show_console=True,
)

recog_fits = {}
recog_idatas = {}

for spec in ddm_specs:
    key = spec["key"]
    stan_path = STAN_DIR / spec["stan_file"]
    data = build_stan_data(spec["extra_keys"])
    print(f"=== Fitting {key} ({stan_path.name}) ===")
    model = CmdStanModel(stan_file=str(stan_path))
    fit = model.sample(data=data, **SAMPLER_KWDS)
    recog_fits[key] = fit
    idata = az.from_cmdstanpy(fit, log_likelihood="log_lik")
    recog_idatas[key] = idata
    nc_path = FITS_DIR / f"recog_ddm_{key}_idata.nc"
    idata.to_netcdf(nc_path)
    print(f"Saved {nc_path}")

In [ ]:
# MCMC for all four models runs in the previous cell (`recog_fits`, `recog_idatas`).

In [ ]:
# Posteriors are built and saved in the fit loop (`recog_idatas`, `fits/recog_ddm_*_idata.nc`).

In [ ]:
import arviz as az
from IPython.display import display

BASE_VARS = ["mu_v", "mu_a", "mu_t0", "sigma_v", "sigma_a", "sigma_t0"]
SUMMARY_VARS = {
    "vanilla": BASE_VARS,
    "reward_drift": BASE_VARS + ["mu_b_reward", "sigma_b_reward"],
    "rpe_drift": BASE_VARS + ["mu_b_rpe", "sigma_b_rpe"],
    "pm_drift": BASE_VARS + ["mu_b_pm", "sigma_b_pm"],
}

for key in SUMMARY_VARS:
    print(f"\n--- az.summary: {key} ---")
    display(az.summary(recog_idatas[key], var_names=SUMMARY_VARS[key]))

# Example: trial-wise drift posterior mean (vanilla DDM)
# drift_trial = recog_idatas["vanilla"].posterior["drift_trial"]
# drift_mean_per_trial = drift_trial.mean(dim=["chain", "draw"]).values


In [ ]:
# Optional: full posterior summary for one model (large output)
# az.summary(recog_idatas["vanilla"])

In [ ]:
# Optional: reload posteriors from disk (skip MCMC). Set True once NetCDF files exist under fits/.
RELOAD_IDATA_FROM_DISK = False

from pathlib import Path

import arviz as az

if RELOAD_IDATA_FROM_DISK:
    fits_dir = Path("fits")
    recog_idatas = {
        k: az.from_netcdf(fits_dir / f"recog_ddm_{k}_idata.nc")
        for k in ("vanilla", "reward_drift", "rpe_drift", "pm_drift")
    }


In [ ]:
import arviz as az

comparison = az.compare(
    {
        "vanilla": recog_idatas["vanilla"],
        "rpe_drift": recog_idatas["rpe_drift"],
        "pm_drift": recog_idatas["pm_drift"],
    },
    ic="loo",
    method="stacking",
    scale="log",
)
print(comparison)

In [ ]:
az.plot_compare(comparison)

In [ ]:
from pathlib import Path

import arviz as az
import numpy as np
import pandas as pd

# Reload vanilla DDM (works in a fresh session after fits were saved)
vanilla_path = Path("fits") / "recog_ddm_vanilla_idata.nc"
if not vanilla_path.is_file():
    raise FileNotFoundError(
        f"Expected saved vanilla fit at {vanilla_path.resolve()}. "
        "Run the MCMC fit loop first (from the Notebooks/ working directory)."
    )

vanilla_idata = az.from_netcdf(vanilla_path)

subject_lookup_df = (
    combined_df[["subj_idx", "participant"]]
    .drop_duplicates()
    .sort_values("subj_idx")
    .reset_index(drop=True)
)


In [ ]:
import re


def _vector_var_summary(idata, var_name: str, hdi_prob: float = 0.94) -> pd.DataFrame:
    """One row per subject index for a Stan vector parameter (v, a, t0)."""
    s = az.summary(idata, var_names=[var_name], hdi_prob=hdi_prob, round_to=None)
    pat = re.compile(rf"^{re.escape(var_name)}\[(\d+)\]$")
    rows = []
    for idx_name, row in s.iterrows():
        m = pat.match(str(idx_name))
        if not m:
            continue
        stan_i = int(m.group(1))
        rows.append(
            {
                "stan_i": stan_i,
                f"{var_name}_mean": float(row["mean"]),
                f"{var_name}_hdi_3": float(row["hdi_3%"]),
                f"{var_name}_hdi_97": float(row["hdi_97%"]),
            }
        )
    return pd.DataFrame(rows)


_parts = [_vector_var_summary(vanilla_idata, v) for v in ("v", "a", "t0")]
subject_params_df = _parts[0]
for p in _parts[1:]:
    subject_params_df = subject_params_df.merge(p, on="stan_i", how="outer")

_idx_min = int(subject_params_df["stan_i"].min())
# CmdStan/ArviZ typically use v[1]..v[S] matching Stan 1-based subjects; if v[0]..v[S-1], shift.
subject_params_df["subj_idx"] = (
    subject_params_df["stan_i"] if _idx_min >= 1 else subject_params_df["stan_i"] + 1
)
subject_params_df = subject_params_df.drop(columns=["stan_i"]).merge(
    subject_lookup_df, on="subj_idx", how="left"
)

_col_order = [
    "subj_idx",
    "participant",
    "v_mean",
    "v_hdi_3",
    "v_hdi_97",
    "a_mean",
    "a_hdi_3",
    "a_hdi_97",
    "t0_mean",
    "t0_hdi_3",
    "t0_hdi_97",
]
subject_params_df = subject_params_df[[c for c in _col_order if c in subject_params_df.columns]]


In [ ]:
# Trial-wise signed drift (same order as rows in combined_df / Stan N)
_dt = vanilla_idata.posterior["drift_trial"]
_flat = _dt.values.reshape(_dt.shape[0] * _dt.shape[1], -1)
_drift_mean = _flat.mean(axis=0)
# Match az.summary default HDI endpoints (~3% / 97% for 94% HDI)
_drift_lo = np.percentile(_flat, 3, axis=0)
_drift_hi = np.percentile(_flat, 97, axis=0)

trial_drift_df = combined_df.reset_index(drop=True).copy()
if trial_drift_df.shape[0] != _drift_mean.shape[0]:
    raise ValueError(
        f"Row mismatch: combined_df has {trial_drift_df.shape[0]} trials but "
        f"drift_trial has {_drift_mean.shape[0]} (Stan N must match dataframe order)."
    )
trial_drift_df["drift_trial_mean"] = _drift_mean
trial_drift_df["drift_trial_hdi_3"] = _drift_lo
trial_drift_df["drift_trial_hdi_97"] = _drift_hi


In [ ]:
from IPython.display import display

display(subject_params_df.head())
display(trial_drift_df.head())


In [ ]:
subject_params_df